In [8]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import date, timedelta
from garmin_data import connect_garmin, get_activities, get_sleep

load_dotenv(override=True)

load_dotenv(override=True)

# Verify credentials loaded
email = os.getenv("GARMIN_EMAIL")
password = os.getenv("GARMIN_PASSWORD")
print(f"Email loaded: {email}")
print(f"Password loaded: {'Yes' if password else 'No'}")


Email loaded: None
Password loaded: No


In [11]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path('/Users/ginamancuso/health-llm/.env')
load_dotenv(dotenv_path=env_path, override=True)

email = os.getenv("GARMIN_EMAIL")
password = os.getenv("GARMIN_PASSWORD")
oura_token = os.getenv("OURA_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

print(f"Garmin email: {email}")
print(f"Garmin password: {'Yes' if password else 'No'}")
print(f"Oura token: {'Yes' if oura_token else 'No'}")
print(f"Anthropic key: {'Yes' if anthropic_key else 'No'}")

Garmin email: gina.mancuso82@gmail.com
Garmin password: Yes
Oura token: Yes
Anthropic key: Yes


In [12]:
# Connect to Garmin with MFA

from garminconnect import Garmin 

email = os.getenv("GARMIN_EMAIL")
password = os.getenv("GARMIN_PASSWORD")

def get_mfa():
    return input("Enter Garmin MFA code: ")

client = Garmin(email, password, prompt_mfa=get_mfa)
client.login()
print("Connected to Garmin!")   

# Oura headers

oura_token = os.getenv("OURA_API_KEY")
headers = {"Authorization": f"Bearer {oura_token}"}

start_date = "2026-05-01"
end_date = date.today().isoformat()


mobile+cffi returned 429: Mobile login returned 429 — IP rate limited by Garmin
mobile+requests returned 429: Mobile login returned 429 — IP rate limited by Garmin


Connected to Garmin!


In [18]:
import requests
from datetime import date

start_date = "2026-05-01"
end_date = date.today().isoformat()

endpoints = {
    "readiness": "daily_readiness",
    "sleep": "daily_sleep",
    "activity": "daily_activity",
    "workout": "workout",
    "stress": "daily_stress",
    "hrv": "daily_hrv"
}

oura_data = {}
for name, endpoint in endpoints.items():
    r = requests.get(
        f"https://api.ouraring.com/v2/usercollection/{endpoint}",
        headers=headers,
        params={"start_date": start_date, "end_date": end_date}
    )
    data = r.json().get('data', [])
    oura_data[name] = data
    print(f"{name}: {len(data)} records")

readiness: 0 records
sleep: 0 records
activity: 0 records
workout: 0 records
stress: 0 records
hrv: 0 records


In [19]:
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv(dotenv_path=Path('/Users/ginamancuso/health-llm/.env'), override=True)
oura_token = os.getenv("OURA_API_KEY")
headers = {"Authorization": f"Bearer {oura_token}"}
print(f"Token: {oura_token[:15] if oura_token else 'MISSING'}")

# Test immediately
import requests
r = requests.get(
    "https://api.ouraring.com/v2/usercollection/personal_info",
    headers=headers
)
print(f"Status: {r.status_code}")
print(r.json())

Token: ORNJGAJTP5Q47VU
Status: 200
{'id': '515891567a162--342cbf0ea69bad62d9f5-', 'age': 43, 'weight': 64.4, 'height': 1.7, 'biological_sex': 'female', 'email': 'gina.mancuso82@gmail.com'}


In [20]:
start_date = "2026-05-01"
end_date = date.today().isoformat()

endpoints = {
    "readiness": "daily_readiness",
    "sleep": "daily_sleep",
    "activity": "daily_activity",
    "workout": "workout",
    "stress": "daily_stress",
    "hrv": "daily_hrv"
}

oura_data = {}
for name, endpoint in endpoints.items():
    r = requests.get(
        f"https://api.ouraring.com/v2/usercollection/{endpoint}",
        headers=headers,
        params={"start_date": start_date, "end_date": end_date}
    )
    data = r.json().get('data', [])
    oura_data[name] = data
    print(f"{name}: {len(data)} records")
    if data:
        print(f"  Sample keys: {list(data[0].keys())}")

readiness: 8 records
  Sample keys: ['id', 'contributors', 'day', 'score', 'temperature_deviation', 'temperature_trend_deviation', 'timestamp']
sleep: 8 records
  Sample keys: ['id', 'contributors', 'day', 'score', 'timestamp']
activity: 7 records
  Sample keys: ['id', 'active_calories', 'average_met_minutes', 'class_5_min', 'contributors', 'day', 'equivalent_walking_distance', 'high_activity_met_minutes', 'high_activity_time', 'inactivity_alerts', 'low_activity_met_minutes', 'low_activity_time', 'medium_activity_met_minutes', 'medium_activity_time', 'met', 'meters_to_target', 'non_wear_time', 'resting_time', 'score', 'sedentary_met_minutes', 'sedentary_time', 'steps', 'target_calories', 'target_meters', 'timestamp', 'total_calories']
workout: 7 records
  Sample keys: ['id', 'activity', 'calories', 'day', 'distance', 'end_datetime', 'intensity', 'label', 'source', 'start_datetime']
stress: 8 records
  Sample keys: ['id', 'day', 'day_summary', 'recovery_high', 'stress_high']
hrv: 0 reco

In [21]:
# Convert each to a DataFrame so we can see it clearly
for name, data in oura_data.items():
    if data:
        df = pd.DataFrame(data)
        print(f"\n{'='*50}")
        print(f"{name.upper()} DATA")
        print(f"{'='*50}")
        print(df.to_string())


READINESS DATA
                                     id                                                                                                                                                                                                               contributors         day  score  temperature_deviation  temperature_trend_deviation                      timestamp
0  d3a70d46-6548-4b09-aba8-e843c42ec554   {'activity_balance': 69, 'body_temperature': 88, 'hrv_balance': None, 'previous_day_activity': 94, 'previous_night': 88, 'recovery_index': 100, 'resting_heart_rate': 87, 'sleep_balance': None, 'sleep_regularity': 84}  2026-05-01     87                   0.21                          NaN  2026-05-01T00:00:00.000+00:00
1  91c02e6b-462b-45de-a76a-f4c1ae7755f9  {'activity_balance': 64, 'body_temperature': 76, 'hrv_balance': None, 'previous_day_activity': 75, 'previous_night': 100, 'recovery_index': 100, 'resting_heart_rate': 34, 'sleep_balance': None, 'sleep_regularity': 84}  20

In [22]:
# Full Oura workout data
print("OURA WORKOUTS:")
df_oura_workout = pd.DataFrame(oura_data['workout'])
print(df_oura_workout[['day', 'activity', 'calories', 'intensity', 'source', 'start_datetime']].to_string())

print("\n\nGARMIN ACTIVITIES:")
df_garmin = pd.DataFrame(client.get_activities_by_date("2026-05-01", date.today().isoformat()))
print(df_garmin[['startTimeLocal', 'activityName', 'calories', 'averageHR', 'duration']].to_string())

OURA WORKOUTS:
          day activity   calories intensity     source                 start_datetime
0  2026-05-03  walking  54.905643  moderate  confirmed  2026-05-03T19:05:00.000-04:00
1  2026-05-04  walking  25.241941  moderate  confirmed  2026-05-04T14:23:00.000-04:00
2  2026-05-04  walking  50.107079  moderate  confirmed  2026-05-04T19:12:00.000-04:00
3  2026-05-05  walking  42.226532  moderate  confirmed  2026-05-05T11:57:00.000-04:00
4  2026-05-06  walking  55.922554  moderate  confirmed  2026-05-06T13:25:00.000-04:00
5  2026-05-07  walking  34.271931  moderate  confirmed  2026-05-07T06:33:00.000-04:00
6  2026-05-07  walking  29.586664  moderate  confirmed  2026-05-07T19:28:00.000-04:00


GARMIN ACTIVITIES:
        startTimeLocal    activityName  calories  averageHR     duration
0  2026-05-08 06:49:10   Indoor Rowing     164.0      126.0  1141.457031
1  2026-05-08 06:11:00      Elliptical     426.0      149.0  2204.364014
2  2026-05-06 05:59:32  Indoor Cycling     589.0      145